# Parte A — Gestión de un trade en vivo sobre IBKR con MAE en R

**Contrato:** `MBT` (Micro Bitcoin, 0.1 BTC) o `BRR` (5 BTC), front month resuelto
programáticamente con `qsCrypto.ibkr.contracts.front_month` — nunca un vencimiento a mano.
**Granularidad:** 30 min, la misma que el backtest (`Turtle_IBKR_CME_Bitcoin.ipynb`).
**Riesgo:** `R = 1N`, con N = EMA-20 del True Range sobre barras de 30 min RTH.

Este notebook hace lo que pide el *Required* de la Parte A, en ese orden:

1. Conectar a TWS/Gateway paper.
2. Calcular ATR y **N**.
3. Lanzar la entrada **con stop adjunto a 1N** (bracket).
4. Calcular y monitorizar **en vivo**: P&L en R, **MAE en R y en precio**, MFE, tiempo en trade
   y tamaño actual — a terminal, a `live_state.json` y al journal.
5. Explicar la aritmética del riesgo (celda markdown de §5).
6. Escribir el **trade journal** con cada decisión.

Y además el mecanismo que hace que esto sea operable y no una demo de juguete: reconciliación
tras caída, fills parciales medidos en R, y un *kill switch*.

---

## El interruptor `INSTRUMENT`: cómo se prueba esto con el mercado cerrado

CME cripto no cotiza 24/7, y este notebook solo se ejercita de verdad contra un mercado que se
mueva: sin ticks no hay MAE, sin fills no hay reconciliación y el monitor no llega a evaluar una
sola regla. La celda de parámetros expone por eso un interruptor `INSTRUMENT`:

| `INSTRUMENT` | Contrato | Barras | Barrera vertical | Tamaño de unidad |
|---|---|---|---|---|
| `"MBT"` | Futuro CME front month | 30 min RTH, `TRADES` | 10 barras = 5 h | sizing Turtle (1% del equity por unidad) |
| `"EURUSD"` | `EUR.USD` en IDEALPRO | 5 min 24 h, `MIDPOINT` | 10 barras ≈ 50 min | **fijo, 25.000 EUR** |

Lo que el modo forex valida: la conexión, la resolución del contrato, la descarga de barras, el
cálculo de N, la aritmética de sizing, el bracket con stop adjunto, la reconciliación tras caída,
las cuatro reglas del monitor, el kill switch y el journal. Es decir, **toda la fontanería**.

Lo que NO valida, y conviene decirlo antes de que alguien lea un número de estos como si fuera un
resultado: nada de la estrategia. `MAE_STOP_N = 0.85`, `TAKE_N = 3` y toda la calibración de §5
salen de excursiones de futuros de Bitcoin. Un par de divisas tiene otra distribución de N, otro
spread y otra estructura de sesión; un trade de EUR.USD que toque el objetivo de 3N no dice
absolutamente nada sobre si la regla funciona en MBT. El tamaño en forex es además un tamaño de
prueba fijo, no sizing Turtle, así que el invariante de "1 unidad = 1% del equity al stop"
tampoco se cumple en ese modo (queda en ~$37 por unidad en vez de ~$2.750).

---

## ⚠️ Lo que el backtest ya dijo, y que condiciona esta sesión

El notebook de Parte B mide, sobre trades cerrados solo por la barrera vertical, que **la MAE
discrimina con claridad**: `P(ganador | MAE ≥ x)` cae de forma monótona con la excursión adversa,
con separación KS de 0.55.

Calibrado con un stop de 2N, **`MAE* = 1.38 N`, por encima del stop duro de 1 N**. Con `R = 1N` no hay hueco para un cierre temprano — el stop dispara antes. Con `R = 2N` el mismo umbral vale 0.69 R y sí lo hay. La regla
se implementa y se opera igual porque el enunciado la pide, pero con la evidencia en contra
escrita por delante.

El otro hallazgo que cambia el orden de magnitud de todo: modelar los **huecos de entrada** (una
ruptura que abre ya pasada el nivel se llena en la apertura, no en el nivel) hunde el Sharpe del
sistema de 3.0 a −0.98. El 25% de las entradas abren con hueco, con 1.21 N de deslizamiento
mediano. El backtest lo resuelve exigiendo una **barra de confirmación** (la siguiente vela
tiene que CERRAR al otro lado del canal, y se entra a ese cierre): el hueco de entrada
desaparece del todo, a cambio de que solo confirme el 27% de las rupturas. La entrada de
este notebook es discrecional a mercado, así que ese filtro no aplica aquí — pero el coste
de cruzar el spread sí, y es el mismo. En vivo eso no es una corrección opcional: es lo que va a pasar.

## 1. Conexión a TWS

In [ ]:
# pip install ib_async pandas numpy matplotlib
import json, math, os, sys, time
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd

# Definición de las rutas relativas del proyecto
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Interactive Brokers API
from ib_async import (IB, ExecutionFilter, Forex, Future, MarketOrder,
                       StopOrder, util)
# Configuraciones de conexión y resolución de contratos
from qsCrypto.ibkr import config
from qsCrypto.ibkr.contracts import front_month
# Misma definición de excursión que usa el backtest. Que Parte A y Parte B midan
# la MAE con EL MISMO código es lo que hace comparable el test de réplica.
from qsCrypto.portfolio.mae import Excursion

# Path del journal
JOURNAL_DIR = ROOT / "journals"; JOURNAL_DIR.mkdir(exist_ok=True)
STATE_DIR = ROOT / "output"; STATE_DIR.mkdir(exist_ok=True)


# ── OBLIGATORIO EN JUPYTER ────────────────────────────────────────────────
# ib_async arranca su propio bucle de eventos y el kernel ya tiene uno: sin
# parchear, `connect()` muere con "This event loop is already running" y el
# notebook degrada en silencio.
def _in_jupyter():
    try:
        return get_ipython().__class__.__name__ == "ZMQInteractiveShell"
    except NameError:
        return False

if _in_jupyter():
    util.startLoop()

# Manejo errores API
IB_ERRORS = []
def _on_ib_error(reqId, code, msg, contract):
    if code not in (2104, 2106, 2158, 2119):   # avisos de granja de datos
        IB_ERRORS.append((datetime.now(timezone.utc).isoformat(), reqId, code, msg))
        print(f"  [TWS {code}] {msg}")

ib = IB()
ib.connect(config.IB_HOST, config.IB_PORT, clientId=config.IB_CLIENT_ID,
           timeout=15, readonly=False)          # readonly=False: aquí SÍ se opera
ib.errorEvent += _on_ib_error

# IB.RequestTimeout viene a 0 y util.run hace `if timeout:` — o sea, CERO
# SIGNIFICA SIN LÍMITE. Cualquier petición bloqueante que no reciba su mensaje
# de fin (reqAccountUpdates sobre una subcuenta FA, por ejemplo) cuelga el
# kernel para siempre y hay que matarlo a mano. Con la sesión en vivo eso deja
# la posición sin vigilar. Se pone un límite real.
ib.RequestTimeout = 20
ib.reqMarketDataType(config.IB_MARKET_DATA_TYPE)

# Careful with this !!!!
ACCOUNT = config.IB_ACCOUNT or ib.managedAccounts()[0]

# ── Cuentas: un dict iterable, no una sola etiqueta ───────────────────────
# Este login es un FA demo: DF* es el master (rechaza órdenes sin reparto) y
# DU* las operables. Una constante ACCOUNT sola no basta, porque un fill puede
# reportarse bajo una etiqueta distinta de la que se pidió y entonces "no veo
# la posición" se confunde con "no hay posición". Se guarda el mapa completo,
# se itera sobre él, y cada fill se atribuye a la cuenta que IBKR diga.
ACCOUNTS = {a: {"id": a,
                "role": "master" if a.upper().startswith("DF") else "trading",
                "monitored": a == ACCOUNT}
            for a in ib.managedAccounts()}
if ACCOUNT not in ACCOUNTS:
    ACCOUNTS[ACCOUNT] = {"id": ACCOUNT, "role": "configured", "monitored": True}
    print(f"  ⚠ IB_ACCOUNT={ACCOUNT!r} NO está en managedAccounts "
          f"{ib.managedAccounts()}: las órdenes se rechazarán o se repartirán a otra.")
print(f"  Cuentas gestionadas  : "
      f"{ {a: v['role'] + ('*' if v['monitored'] else '') for a, v in ACCOUNTS.items()} }")
SRV_TIME = pd.Timestamp(ib.reqCurrentTime())     # una sola vez por sesión
IB_TIME_OFFSET = SRV_TIME - pd.Timestamp.now(tz="UTC")

print(f"  Conectado a {config.IB_HOST}:{config.IB_PORT} (clientId={config.IB_CLIENT_ID})")
print(f"  Cuenta operada       : {ACCOUNT}")
print(f"  Hora servidor        : {SRV_TIME:%Y-%m-%d %H:%M:%S %Z}")
print(f"  Tipo de datos        : {config.IB_MARKET_DATA_TYPE} "
      f"({'RETRASADOS' if config.IB_MARKET_DATA_TYPE == 3 else 'en tiempo real'})")
if config.IB_MARKET_DATA_TYPE == 3:
    print("  ⚠ Con datos retrasados (15 min) la MAE se mide sobre precios viejos: el")
    print("    cierre temprano reacciona tarde y la MAE registrada infravalora la real.")
    print("    Aceptable para el ejercicio, inaceptable con dinero de verdad.")

summary = {v.tag: v.value for v in ib.accountSummary(ACCOUNT)
           if v.tag in ("NetLiquidation", "AvailableFunds", "BuyingPower")}
print(f"  Resumen de cuenta    : {summary}")
EQUITY = float(summary.get("NetLiquidation", 275_000))

## 2. Contrato, ATR y N

`N` es la EMA-20 del True Range sobre barras de **30 min RTH**, exactamente la misma fórmula y
la misma granularidad que el backtest. Cambiar cualquiera de las dos rompería la comparación
entre lo que se simuló y lo que se opera.

In [ ]:
# ─────────── INSTRUMENTO: futuro CME (producción) o forex (prueba) ───────────
# Los futuros CME de cripto no cotizan 24/7, y este notebook solo se ejercita de
# verdad contra un mercado que se mueva: sin ticks no hay MAE, sin fills no hay
# reconciliación y el monitor no evalúa una sola regla. EUR.USD en IDEALPRO cotiza
# ~24/5 en la MISMA cuenta paper, así que sirve de banco de pruebas de la fontanería.
# No valida NADA de la estrategia: MAE_STOP_N, TAKE_N y la calibración de §5 salen
# de excursiones de futuros de Bitcoin, no de un par de divisas.
INSTRUMENT = "EURUSD"        # "MBT" = producción (futuro CME) | "EURUSD" = prueba

INSTRUMENTS = {
    "MBT": dict(
        kind="future", symbol="MBT", exchange="CME", currency="USD",
        bar_size="30 mins", bar_seconds=30 * 60, use_rth=True, what_to_show="TRADES",
        max_hold_bars=10, session_hours=6.0,
        qty_step=1, qty_test=None,          # sin tope: manda el sizing Turtle
        market_data_type=None,              # el de config
        stop_n=1.0, mae_stop_n=0.85, max_units=4,
        log_every=6, enter_on_reconciled=False,
    ),
    "EURUSD": dict(
        kind="forex", symbol="EURUSD", exchange="IDEALPRO", currency="USD",
        # 5 min en vez de 30: la barrera vertical cae a ~50 min y una sesión de
        # prueba completa (entrada → add / MAE / objetivo / vertical) cabe en media
        # hora. Es el único parámetro que se toca por prisa, y se toca explícitamente.
        bar_size="5 mins", bar_seconds=5 * 60, use_rth=False, what_to_show="MIDPOINT",
        max_hold_bars=10, session_hours=0.5,
        # IDEALPRO cotiza en unidades de la divisa base. 25.000 EUR es un TAMAÑO DE
        # PRUEBA FIJO, no sizing Turtle: ver la celda de sizing para la aritmética
        # y para lo que se rompe al clavarlo.
        qty_step=1_000, qty_test=25_000,
        # Stop a 2N y cierre temprano cuando la MAE llega a 1R. Como R = STOP_N·N,
        # 1R = 2N = exactamente donde está el stop del bróker: las dos reglas caen
        # en el mismo precio y se disputan quién llega antes. Es lo pedido y se
        # deja así, pero para que el cierre por MAE muerda ANTES que el stop hay
        # que bajar mae_stop_n (el backtest sitúa MAE* en 1.38N = 0.69R).
        stop_n=2.0, mae_stop_n=2.0,
        max_units=1,          # sin escalera: la prueba es una entrada, no un add
        log_every=1,          # registro en CADA poll, no cada 6
        enter_on_reconciled=True,   # entra ENCIMA de lo que reconcilie
        # IDEALPRO no publica datos retrasados: con marketDataType=3 el ticker llega
        # vacío y el monitor se quedaría girando sobre NaN. Se pide tiempo real.
        market_data_type=1,
    ),
}
if INSTRUMENT not in INSTRUMENTS:
    raise KeyError(f"INSTRUMENT={INSTRUMENT!r} no está en {list(INSTRUMENTS)}")
CFG = INSTRUMENTS[INSTRUMENT]

# ─────────── PARÁMETROS, DECLARADOS ANTES DE ABRIR NADA ───────────
KIND          = CFG["kind"]
SYMBOL        = CFG["symbol"]      # clave del journal y del live_state
BAR_SIZE      = CFG["bar_size"]
BAR_SECONDS   = CFG["bar_seconds"]
USE_RTH       = CFG["use_rth"]
WHAT_TO_SHOW  = CFG["what_to_show"]
QTY_STEP      = CFG["qty_step"]
QTY_TEST      = CFG["qty_test"]
ATR_PERIOD    = 20

RISK_PER_UNIT = 0.01      # 1% del equity por unidad
STOP_N        = CFG["stop_n"]          # R = STOP_N·N. Stop duro en el bróker.
TAKE_N        = 3.0       # barrera superior
MAX_HOLD_BARS = CFG["max_hold_bars"]   # barrera vertical, en barras de BAR_SIZE
ADD_N         = 0.5       # escalera de adds
MAX_UNITS     = CFG["max_units"]
MAE_STOP_N    = CFG["mae_stop_n"]      # cierre temprano — ver §5
MAX_GROSS_LEV = 4.0
LOG_EVERY     = CFG["log_every"]       # cada cuántos polls se registra
ENTER_ON_RECONCILED = CFG["enter_on_reconciled"]

SIDE          = "BUY"     # "BUY" | "SELL" — la entrada es discrecional (§3)
POLL_SECONDS  = 5         # cadencia del monitor
SESSION_HOURS = CFG["session_hours"]   # duración máxima de la monitorización

# MBT en vez de BRR: 0.1 BTC por contrato da granularidad suficiente para que la
# escalera de 4 unidades exista de verdad. Con BRR (5 BTC) y este capital solo
# caben 1-2 contratos y la lógica de adds no llegaría a ejercitarse.

if KIND == "future":
    contract, detail, expiry = front_month(ib, SYMBOL, CFG["exchange"], CFG["currency"])
    MULT = float(contract.multiplier)
else:
    # Un par al contado no tiene cadena de vencimientos: se cualifica y ya. Se usa
    # Forex() en vez de front_month() precisamente porque no hay front month.
    contract = Forex(SYMBOL)                      # "EURUSD" → EUR.USD @ IDEALPRO
    if not ib.qualifyContracts(contract):
        raise RuntimeError(f"IBKR no resolvió {SYMBOL} como par de divisas.")
    details = ib.reqContractDetails(contract)
    if not details:
        raise RuntimeError(f"Sin contract details para {SYMBOL}@{CFG['exchange']}.")
    detail, expiry = details[0], None
    MULT = 1.0      # el "multiplicador" de un par al contado es 1 unidad de la base

TICK = float(detail.minTick)
# Decimales del tick: 5e-05 → 5. Vía Decimal y no log10 porque log10(5e-05) = -4.30
# redondea a 4 y dejaría los stops de forex a 4 decimales, que TWS rechaza.
PX_DEC = max(0, -Decimal(str(TICK)).as_tuple().exponent)

def fmt_px(px):
    """Precio con los decimales del tick. Sin esto, EUR.USD se imprime como '1.1'."""
    if px is None or not np.isfinite(px):
        return "nan"
    return f"{px:,.{PX_DEC}f}"

# El tipo de dato de mercado puede depender del instrumento (forex no tiene retrasado)
MKT_DATA_TYPE = CFG["market_data_type"] or config.IB_MARKET_DATA_TYPE
ib.reqMarketDataType(MKT_DATA_TYPE)

print(f"  Instrumento: {INSTRUMENT}  ({KIND})")
print(f"  Contrato   : {contract.localSymbol}  conId={contract.conId}")
if expiry is not None:
    print(f"  Vencimiento: {expiry}   (resuelto programáticamente, no hardcodeado)")
else:
    print(f"  Vencimiento: n/a (contado)")
print(f"  Multiplier : {MULT}   minTick: {TICK}  → valor de tick ${TICK*MULT:,.2f}")
print(f"  Barras     : {BAR_SIZE} ({WHAT_TO_SHOW}, useRTH={USE_RTH}) → "
      f"vertical a {MAX_HOLD_BARS} barras = {MAX_HOLD_BARS*BAR_SECONDS/3600:.1f} h")
print(f"  Datos      : marketDataType={MKT_DATA_TYPE}")
print(f"  Riesgo     : stop {STOP_N:.0f}N (= 1R) | cierre MAE a {MAE_STOP_N:.2f}N "
      f"= {MAE_STOP_N/STOP_N:.2f}R | objetivo {TAKE_N:.0f}N = {TAKE_N/STOP_N:.2f}R")
if abs(MAE_STOP_N / STOP_N - 1.0) < 1e-9:
    print("  ⚠ El cierre por MAE (1R) cae en el MISMO precio que el stop del bróker")
    print("    (1R = STOP_N·N). Las dos reglas compiten; ganará la que llegue antes.")
print(f"  Horario líquido: {str(detail.liquidHours)[:80]}")

if INSTRUMENT != "MBT":
    print("\n  " + "=" * 72)
    print("  MODO PRUEBA — instrumento sustituto, NO la estrategia.")
    print("  Valida: conexión, contrato, barras, N, sizing, bracket, reconciliación,")
    print("  las 4 reglas del monitor, kill switch y journal. Es decir, la fontanería.")
    print("  NO valida: nada del edge. MAE_STOP_N=0.85 y TAKE_N=3 se calibraron sobre")
    print("  excursiones de futuros de Bitcoin; un resultado aquí no se traslada a MBT.")
    print("  " + "=" * 72)


In [ ]:
CFG

In [ ]:
def compute_n(df, period=ATR_PERIOD):
    """N = EMA-{period} del True Range. Fórmula Turtle exacta, idéntica al backtest."""
    h, l, c = df["high"].values, df["low"].values, df["close"].values
    pdc = np.roll(c, 1); pdc[0] = c[0]
    tr = np.maximum(h - l, np.maximum(np.abs(h - pdc), np.abs(pdc - l)))
    n = np.full(len(tr), np.nan)
    if len(tr) < period:
        return n
    n[period - 1] = tr[:period].mean()
    for i in range(period, len(tr)):
        n[i] = ((period - 1) * n[i - 1] + tr[i]) / period
    return n


def fetch_bars(dur="5 D"):
    bars = ib.reqHistoricalData(contract, endDateTime="", durationStr=dur,
                                barSizeSetting=BAR_SIZE, whatToShow=WHAT_TO_SHOW,
                                useRTH=USE_RTH, formatDate=2, timeout=60)
    if not bars:
        raise RuntimeError(
            f"reqHistoricalData no devolvió barras para {contract.localSymbol} "
            f"({dur}, {BAR_SIZE}, {WHAT_TO_SHOW}, useRTH={USE_RTH}). "
            "En divisas whatToShow='TRADES' devuelve vacío: usa MIDPOINT.")
    d = util.df(bars).rename(columns={"date": "time"})
    d["time"] = pd.to_datetime(d["time"], utc=True).dt.tz_localize(None)
    d = d.set_index("time").sort_index()
    # Las barras MIDPOINT no traen volumen (IB devuelve -1). Se descarta la columna
    # en vez de arrastrar un número que miente; compute_n solo usa high/low/close.
    cols = ["open", "high", "low", "close"]
    if "volume" in d.columns and (d["volume"] > 0).any():
        cols.append("volume")
    return d[cols].astype(float)


BARS = fetch_bars()
BARS["n"] = compute_n(BARS)
N = float(BARS["n"].iloc[-1])
PX = float(BARS["close"].iloc[-1])
if not np.isfinite(N) or N <= 0:
    raise RuntimeError(f"N no es finito ({N}): {len(BARS)} barras para un ATR de "
                       f"{ATR_PERIOD}. Amplía la duración de fetch_bars().")

# Sizing Turtle: una unidad que se mueve 1N produce exactamente RISK_PER_UNIT
# del equity. Los contratos son ENTEROS, así que se trunca hacia abajo: redondear
# hacia arriba excedería el riesgo previsto en silencio.
UNIT_QTY_TURTLE = math.floor((RISK_PER_UNIT * EQUITY) / (N * MULT))

# El clamp de prueba. En forex el sizing Turtle da ~1,8M EUR por unidad (N vale
# ~0.0015 y el multiplicador es 1): 6-7x el equity por unidad y ~27x con las cuatro,
# que ni es operable ni deja ejercitar la escalera de adds. Se fija un tamaño de
# prueba y se redondea al paso del lote. La cifra Turtle se sigue calculando y se
# imprime: lo que se sustituye es el tamaño, no la aritmética.
if QTY_TEST is None:
    UNIT_QTY = UNIT_QTY_TURTLE
else:
    UNIT_QTY = int(min(UNIT_QTY_TURTLE, QTY_TEST) // QTY_STEP * QTY_STEP)

R_PTS = STOP_N * N                       # 1R en puntos de precio
RISK_USD = UNIT_QTY * R_PTS * MULT       # riesgo en USD de UNA unidad al stop

print(f"  Barras          : {len(BARS)}  ({BARS.index[0]:%m-%d %H:%M} → {BARS.index[-1]:%m-%d %H:%M})")
print(f"  Precio          : {fmt_px(PX)}")
print(f"  N (EMA-{ATR_PERIOD} TR)   : {fmt_px(N)}  = {N/PX:.2%} del precio")
print(f"  R = {STOP_N:.0f}N        : {fmt_px(R_PTS)}")
print(f"  Unidad Turtle   : {UNIT_QTY_TURTLE:,} → riesgo "
      f"${UNIT_QTY_TURTLE*R_PTS*MULT:,.0f} ({RISK_PER_UNIT:.0%} del equity, por construcción)")
print(f"  Unidad OPERADA  : {UNIT_QTY:,}  → riesgo ${RISK_USD:,.0f} "
      f"({RISK_USD/EQUITY:.2%} del equity)")
print(f"  Nocional/unidad : ${UNIT_QTY*PX*MULT:,.0f} = {UNIT_QTY*PX*MULT/EQUITY:.2f}x equity")

if QTY_TEST is not None and UNIT_QTY != UNIT_QTY_TURTLE:
    print(f"\n  ⚠ TAMAÑO DE PRUEBA FIJO ({QTY_TEST:,}, paso {QTY_STEP:,}), no sizing Turtle.")
    print(f"    El invariante '1 unidad = {RISK_PER_UNIT:.0%} del equity al stop' NO se cumple")
    print(f"    en este modo: {RISK_USD/EQUITY:.3%} en vez de {RISK_PER_UNIT:.0%}. Las lecturas en R")
    print( "    siguen siendo correctas (R es una distancia de precio, no un tamaño), pero")
    print( "    cualquier cifra en USD de esta sesión es de juguete y no escala a MBT.")

# La restricción que decide si esta granularidad es operable, y que no depende
# del capital: nocional/equity = RISK × precio / N.
_lev_teorico = RISK_PER_UNIT * PX / N
_lev = UNIT_QTY * PX * MULT / EQUITY
print(f"\n  Apalancamiento por unidad: {_lev:.2f}x  (Turtle puro daría {_lev_teorico:.2f}x, "
      f"tope {MAX_GROSS_LEV}x)")
assert UNIT_QTY >= 1, (
    f"qty=0: con N={fmt_px(N)} y equity ${EQUITY:,.0f} no cabe ni una unidad. "
    + ("Sube el capital o usa MBT en vez de BRR." if KIND == "future"
       else f"Baja qty_step ({QTY_STEP:,}) para este par."))
if _lev * MAX_UNITS > MAX_GROSS_LEV:
    print(f"  ⚠ Con {MAX_UNITS} unidades se llegaría a {_lev*MAX_UNITS:.1f}x: el tope de "
          f"apalancamiento morderá antes de completar la escalera.")


## 3. Journal y estado — se escriben ANTES de operar

El journal es el entregable, no un log de depuración: cada decisión con su timestamp, precio,
tamaño y la MAE/MFE del momento, más la regla que disparó. Se abre antes de mandar la primera
orden para que no exista ningún estado operado que no esté escrito en disco.

`live_state.json` sigue el patrón del market maker: escritura **atómica** (fichero temporal y
`os.replace`), de modo que un lector nunca vea un JSON a medias.

In [ ]:
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
JOURNAL = JOURNAL_DIR / f"parteA_{SYMBOL}_{SESSION_ID}.jsonl"
STATE = STATE_DIR / f"live_state_{SYMBOL}.json"

def now_srv():
    """Hora del servidor de IB, derivada del desfase medido al conectar."""
    return (pd.Timestamp.now(tz="UTC") + IB_TIME_OFFSET).tz_localize(None)


def journal(evento, regla="", price=None, qty=None, exc=None, **extra):
    """Una línea por decisión. Append-only y con flush: si el proceso muere, lo
    escrito hasta ese instante sigue siendo la verdad reconstruible."""
    row = {
        "ts": now_srv().isoformat(),
        "session": SESSION_ID, "symbol": SYMBOL,
        "contract": contract.localSymbol, "evento": evento, "regla": regla,
        "price": None if price is None else float(price),
        "qty": None if qty is None else int(qty),
        "n": N, "r_pts": R_PTS,
    }
    if exc is not None:
        s = exc.snapshot(price)
        row |= {k: (None if (isinstance(v, float) and not np.isfinite(v)) else v)
                for k, v in s.items()}
    row |= extra
    with open(JOURNAL, "a") as f:
        f.write(json.dumps(row, default=str) + "\n")
        f.flush()
    return row


def write_state(payload):
    """Escritura atómica: tmp + os.replace. Un lector nunca ve un JSON parcial."""
    tmp = STATE.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(payload, f, indent=2, default=str)
    os.replace(tmp, STATE)


journal("session_start", regla="arranque",
        equity=EQUITY, unit_qty=UNIT_QTY, risk_usd=RISK_USD,
        mae_stop_n=MAE_STOP_N, take_n=TAKE_N, max_hold_bars=MAX_HOLD_BARS,
        account=ACCOUNT, market_data_type=config.IB_MARKET_DATA_TYPE)
print(f"  Journal : {JOURNAL}")
print(f"  Estado  : {STATE}")

## 3 bis. Reconciliación — arrancar sobre un estado que ya existe

Si el proceso muere a mitad de trade y se relanza, **no** puede asumir cuenta limpia. Se
reconstruye desde dos fuentes:

- **IBKR** sabe la posición y las órdenes vivas. Es la verdad sobre qué hay puesto.
- **El journal** sabe la excursión. IBKR **no guarda la MAE**

Y queda un hueco entre la última línea del journal y ahora, que se rellena con las barras de 30
min de ese intervalo. **Sin ese *backfill* un reinicio pone la MAE a cero y el cierre temprano
deja de disparar**: el peor fallo posible, porque es silencioso y deja el sistema operando con
una regla que cree activa.

In [ ]:
# ib.portfolio() es una SEGUNDA fuente de posición, independiente del stream de
# reqPositions. NO se llama a ib.reqAccountUpdates(): ib_async ya se suscribe al
# conectar ("This is called at startup - no need to call again") y, sobre este
# login FA, volver a pedirlo para una subcuenta nunca recibe accountDownloadEnd
# y se queda colgado para siempre. Si portfolio viene vacío no pasa nada: es la
# fuente 2 de 3, y para divisas la buena es la 3 (ejecuciones).
_WARNED = set()

# Qué fuentes respondieron en la última consulta. Importa porque una fuente que
# FALLA y una que dice "no hay posición" son cosas distintas, y confundirlas es
# exactamente el error que abandonó tres posiciones vivas el 2026-09-07.
_SRC_OK = {"positions": True, "portfolio": True, "executions": True}

# Por debajo de esto es residuo contable, no posición: las subcuentas arrastran
# saldos de céntimos (-0.06 EUR) de operaciones viejas.
FX_MIN_UNIT = 1.0


def fx_positions():
    """{cuenta: {...}} de una divisa AL CONTADO. No usa reqPositions.

    IBKR no publica las posiciones spot ni por reqPositions ni por portfolio.
    Medido el 2026-09-07 con 25.000 EUR vivos: tres pasadas de reqPositions en
    9 s no devolvieron una sola fila con cantidad ≠ 0 sobre conId 12087792, y
    ib.portfolio() salió vacío en las seis cuentas. Una posición al contado es
    un SALDO EN DIVISA — en TWS vive en el FX Portfolio —, no un objeto Position.

    Se combinan las dos únicas fuentes que sí la ven:

    * el neto de ejecuciones de hoy, que es INMEDIATO (llega con el fill) y trae
      además el precio medio de entrada, que el saldo no da;
    * el saldo de caja en la divisa base, que va con retraso (IBKR refresca los
      valores de cuenta cada pocos minutos) pero recoge lo que las ejecuciones
      de hoy no pueden ver: una posición heredada de días anteriores.

    Manda la ejecución cuando existe, y el saldo cuando no hay operaciones hoy.
    Ese orden es el que evita los dos fallos: el saldo retrasado diría "plano"
    justo después de entrar, y el neto de ejecuciones diría "plano" con una
    posición traída de ayer.
    """
    base = contract.symbol                      # "EUR" en EUR.USD
    st = exec_state()
    out = {}
    try:
        for acct, meta in ACCOUNTS.items():
            if meta["role"] == "master":        # el master publica agregados
                continue
            bal = 0.0
            for v in ib.accountValues(acct):
                if v.tag == "CashBalance" and v.currency == base:
                    bal = float(v.value); break
            e = st.get(acct, {})
            e_qty, e_avg = float(e.get("qty", 0.0)), float(e.get("avg", 0.0))
            qty = e_qty if abs(e_qty) >= FX_MIN_UNIT else bal
            if abs(qty) < FX_MIN_UNIT:
                continue
            src = "executions" if abs(e_qty) >= FX_MIN_UNIT else "cash"
            out[acct] = {"account": acct, "position": qty, "avgCost": e_avg,
                         "source": src}
            if (abs(bal) >= FX_MIN_UNIT and abs(e_qty) >= FX_MIN_UNIT
                    and abs(bal - e_qty) > FX_MIN_UNIT
                    and f"fxdiff_{acct}" not in _WARNED):
                _WARNED.add(f"fxdiff_{acct}")
                print(f"  ⚠ {acct}: saldo {base} = {bal:,.2f} vs neto de ejecuciones "
                      f"{e_qty:,.2f} (dif {bal - e_qty:+,.2f}). Puede ser el retraso "
                      f"de los valores de cuenta, o movimientos ajenos a esto.")
        _SRC_OK["positions"] = _SRC_OK["portfolio"] = True
    except Exception as e:
        _SRC_OK["positions"] = False
        print(f"  ⚠ lectura de saldos de caja falló ({type(e).__name__}).")
    return out


def positions_on_contract():
    """{cuenta: {account, position, avgCost, source}} de ESTE contrato.

    Indexado por cuenta y por tanto iterable: el monitor, el kill switch y la
    reconciliación necesitan saber no solo CUÁNTO hay vivo sino DÓNDE, porque
    una orden de cierre mandada a la cuenta equivocada no cierra nada.

    Se localiza por conId PRIMERO y por cuenta después, al revés de lo obvio.
    Filtrar por cuenta antes que por contrato es lo que rompió la sesión del
    2026-09-07: la posición estaba viva en TWS pero la lista salía vacía, y el
    monitor leyó "vacía" como "me han saltado el stop". El contrato es un hecho
    del mercado; la etiqueta de cuenta es una convención del bróker que en
    logins FA puede no ser la que se pidió. Se busca por el hecho y luego se
    mira la etiqueta.
    """
    if contract.secType == "CASH":
        return fx_positions()               # el contado no pasa por reqPositions
    out = {}
    try:
        ib.reqPositions()                   # bloqueante: vuelve con positionEnd
        for p in ib.positions():
            if p.contract.conId == contract.conId and abs(p.position) > 1e-9:
                out[p.account] = {"account": p.account, "position": float(p.position),
                                  "avgCost": float(p.avgCost or 0.0),
                                  "source": "positions"}
        _SRC_OK["positions"] = True
    except Exception as e:
        _SRC_OK["positions"] = False
        print(f"  ⚠ reqPositions falló ({type(e).__name__}): esta fuente NO dice "
              f"que no haya posición, dice que no lo sabe.")
    try:
        for it in ib.portfolio():           # 2ª fuente
            if it.contract.conId == contract.conId and abs(it.position) > 1e-9:
                out.setdefault(it.account,
                               {"account": it.account, "position": float(it.position),
                                "avgCost": float(it.averageCost or 0.0),
                                "source": "portfolio"})
        _SRC_OK["portfolio"] = True
    except Exception as e:
        _SRC_OK["portfolio"] = False
        print(f"  ⚠ portfolio falló ({type(e).__name__}).")
    return out


def tradeable_account(preferred=None):
    """Cuenta a la que se pueden MANDAR órdenes. Nunca el master.

    Reportar y operar NO son la misma cuenta en un login FA. Medido el
    2026-09-07 sobre EUR.USD: la posición se reporta bajo DF496956 (el master)
    y las ejecuciones bajo DU496961 (la subcuenta). Mandar el cierre a la
    cuenta que reporta lo haría rechazar por falta de reparto — el kill switch
    creería haber aplanado y la posición seguiría viva.
    """
    for a in (preferred, ACCOUNT):
        if a and ACCOUNTS.get(a, {}).get("role") != "master":
            return a
    for a, v in ACCOUNTS.items():
        if v["role"] == "trading":
            return a
    return ACCOUNT


_EXEC_CACHE = {"t": 0.0, "net": {}, "state": {}}


def exec_state(max_age=20.0):
    """{subcuenta: {qty, avg}} desde las ejecuciones de HOY, en orden temporal.

    Devuelve también el precio medio de entrada. En divisas es el único sitio de
    donde sacarlo: el saldo de caja dice cuánto hay, no a qué precio se entró, y
    reqPositions —que traería avgCost— no reporta el contado.
    """
    if max_age and (time.time() - _EXEC_CACHE["t"]) < max_age:
        return _EXEC_CACHE["state"]
    try:
        execs = ib.reqExecutions(ExecutionFilter())
        _SRC_OK["executions"] = True
    except Exception as e:
        _SRC_OK["executions"] = False
        print(f"  ⚠ reqExecutions falló ({type(e).__name__}): se reutiliza la caché.")
        return _EXEC_CACHE["state"]
    rows = [f for f in execs
            if f.contract.conId == contract.conId
            and ACCOUNTS.get(f.execution.acctNumber, {}).get("role") != "master"]
    rows.sort(key=lambda f: f.execution.time)
    st = {}
    for f in rows:
        s = st.setdefault(f.execution.acctNumber, {"qty": 0.0, "avg": 0.0})
        q = float(f.execution.shares) * (1.0 if f.execution.side == "BOT" else -1.0)
        px, new = float(f.execution.price), s["qty"] + q
        if abs(new) < 1e-9:
            s["avg"] = 0.0                                  # se cierra
        elif abs(s["qty"]) < 1e-9 or (s["qty"] > 0) != (new > 0):
            s["avg"] = px                                   # abre o gira de signo
        elif abs(new) > abs(s["qty"]):
            s["avg"] = (s["avg"] * abs(s["qty"]) + px * abs(q)) / abs(new)
        # si solo reduce, el precio medio de entrada no cambia
        s["qty"] = new
    net = {a: v["qty"] for a, v in st.items() if abs(v["qty"]) > 1e-9}
    _EXEC_CACHE.update(t=time.time(), state=st, net=net)
    return st


def executions_net(max_age=20.0):
    """{subcuenta: cantidad_neta} de HOY para este contrato, según ejecuciones.

    SIRVE PARA ENRUTAR, NO PARA SABER SI HAY POSICIÓN. Las posiciones de contado
    se reportan agregadas bajo el master, pero las ejecuciones sí se atribuyen a
    la subcuenta que operó: eso es lo que dice a qué cuenta hay que mandar el
    cierre. Como medida de posición es un contador que solo sabe sumar lo que
    pasó por la API — un cierre manual en TWS (o una conversión de divisa) no
    deja ejecución sobre este conId, y el neto se queda colgado en la última
    compra. Usarlo como "hay posición" bloqueó el arranque limpio del
    2026-09-07: decía +70.000 con la cuenta ya plana. Las posiciones mandan.

    Se excluye el master porque publica además el agregado de las subcuentas y
    contarlo sería duplicar.
    """
    exec_state(max_age)                    # rellena la caché (y la comparte)
    return _EXEC_CACHE["net"]


def live_position(prefer=None, warn=True):
    """Posición viva, priorizando la cuenta `prefer` (la de la entrada).

    Devuelve (cantidad_firmada, cuenta_que_reporta, cuenta_operable, avgCost).
    Solo devuelve 0 cuando NINGUNA de las TRES fuentes ve nada: posiciones,
    portfolio y ejecuciones del día. Con una sola fuente, "no lo veo" y "no
    está" son indistinguibles, y esa confusión es la que abandonó tres
    posiciones vivas de EUR.USD el 2026-09-07.
    """
    by_acct = positions_on_contract()
    ex_net = executions_net()
    subs = {a: r for a, r in by_acct.items()
            if ACCOUNTS.get(a, {}).get("role") != "master"}
    mstr = {a: r for a, r in by_acct.items()
            if ACCOUNTS.get(a, {}).get("role") == "master"}

    if subs:                      # lo normal: la subcuenta reporta su posición
        key = next((a for a in (prefer, ACCOUNT) if a in subs), next(iter(subs)))
        r = subs[key]
        qty, rep, avg, src = r["position"], key, r["avgCost"], r["source"]
    elif mstr:                    # divisas en FA: solo el master la reporta
        key = next(iter(mstr)); r = mstr[key]
        qty, rep, avg, src = r["position"], key, r["avgCost"], r["source"]
        if warn and "master" not in _WARNED:
            _WARNED.add("master")
            print(f"  ⚠ {contract.localSymbol} lo reporta SOLO el master {key!r} "
                  f"(agregado de subcuentas). Las órdenes NO pueden ir ahí.")
    elif not (_SRC_OK["positions"] or _SRC_OK["portfolio"]):
        # Ninguna fuente de posición respondió. Eso NO es estar plano: es no
        # saberlo. Solo aquí valen las ejecuciones como última pista.
        if ex_net:
            key = next((a for a in (prefer, ACCOUNT) if a in ex_net), next(iter(ex_net)))
            print(f"  ⚠ Las fuentes de posición no responden; las ejecuciones de hoy "
                  f"dejan {ex_net[key]:+,.0f} neto en {key!r}. Se asume viva.")
            return ex_net[key], key, tradeable_account(key), float("nan")
        print("  ⚠ TODAS las fuentes de posición fallaron: estado DESCONOCIDO.")
        return None, None, tradeable_account(prefer), float("nan")
    else:
        # Plano de verdad: las fuentes de posición respondieron y no ven nada.
        # Si las ejecuciones del día no cuadran se avisa, pero NO se bloquea: un
        # cierre manual en TWS no deja rastro en este conId y ese descuadre es
        # lo normal, no una posición fantasma.
        if ex_net and warn and "exec_stale" not in _WARNED:
            _WARNED.add("exec_stale")
            print(f"  ℹ Sin posición reportada, pero las ejecuciones de hoy suman "
                  f"{ {a: round(q) for a, q in ex_net.items()} }.")
            print( "    Normal si cerraste a mano en TWS. Manda la posición, no el contador.")
        return 0.0, None, tradeable_account(prefer), float("nan")

    trade = tradeable_account(prefer if prefer in ex_net else
                              (rep if ACCOUNTS.get(rep, {}).get("role") != "master"
                               else next(iter(ex_net), None)))
    if trade != rep and warn and "split" not in _WARNED:
        _WARNED.add("split")
        print(f"  ⚠ {contract.localSymbol}: reporta {rep!r} (vía {src}) pero se opera "
              f"en {trade!r}. Reportar y operar son cuentas distintas aquí.")
    return qty, rep, trade, (avg / MULT if avg else float("nan"))


def reconcile():
    """Devuelve (posicion_ib, orden_stop_viva, excursion_reconstruida) o (None, None, None)."""
    ib.sleep(1.5)
    qty_signed, acct, trade_acct, avg = live_position()
    open_orders = [o for o in ib.reqOpenOrders()
                   if o.contract.conId == contract.conId]
    stop_orders = [o for o in open_orders if o.order.orderType in ("STP", "STP LMT")]

    if qty_signed is None:
        raise RuntimeError(
            "Ninguna fuente de posición respondió: no se puede arrancar sin saber "
            "si hay algo abierto. Revisa la conexión a TWS y vuelve a ejecutar.")
    if abs(qty_signed) < 1e-9:
        if stop_orders:
            # Un stop vivo sin posición no es inocuo: si salta, ABRE posición
            # contraria. Antes de un arranque limpio hay que barrerlo.
            print(f"  ⚠ {len(stop_orders)} orden(es) STOP vivas SIN posición: son "
                  f"huérfanas y pueden abrir posición contraria si saltan.")
            print( "    Ejecuta kill_switch() para cancelarlas antes de abrir nada.")
        print("  Sin posición abierta en IBKR: arranque limpio.")
        return None, (stop_orders[0] if stop_orders else None), None

    from types import SimpleNamespace
    p = SimpleNamespace(account=trade_acct, report_account=acct,
                        position=qty_signed, avgCost=avg * MULT)
    side = 1 if qty_signed > 0 else -1
    qty = abs(int(qty_signed))
    print(f"  ⚠ POSICIÓN VIVA reconciliada: {p.position:+,.0f} @ {fmt_px(avg)} "
          f"(reporta {acct}, se opera en {trade_acct})")
    print(f"    Órdenes stop vivas en el bróker: {len(stop_orders)}")

    # 1) recuperar del journal la última excursión conocida de esta sesión
    prev, anchor, opened_at = None, avg, None
    for jf in sorted(JOURNAL_DIR.glob(f"parteA_{SYMBOL}_*.jsonl")):
        for line in open(jf):
            try: row = json.loads(line)
            except Exception: continue
            if row.get("evento") in ("entry", "add", "tick") and row.get("mae_price"):
                prev = row
                anchor = row.get("anchor0", anchor)
                opened_at = opened_at or row.get("ts")

    exc = Excursion(side=side, anchor0=float(anchor), n0=N, r0=R_PTS)
    if prev:
        exc.mae_px = float(prev["mae_price"]); exc.mfe_px = float(prev["mfe_price"])
        print(f"    Excursión recuperada del journal: MAE={exc.mae_r:.2f}R "
              f"MFE={exc.mfe_r:.2f}R (último tick {prev['ts']})")
        # 2) rellenar el hueco entre esa marca y ahora con barras reales
        gap_from = pd.Timestamp(prev["ts"])
        recent = fetch_bars("2 D")
        gap = recent[recent.index > gap_from]
        for _, b in gap.iterrows():
            exc.update_bar(b["high"], b["low"])
        if len(gap):
            print(f"    Backfill de {len(gap)} barras desde {gap_from:%H:%M} → "
                  f"MAE={exc.mae_r:.2f}R")
    else:
        print("    Sin journal previo: la excursión arranca en cero y estará SUBESTIMADA.")
    journal("reconcile", regla="arranque_con_posicion", price=avg, qty=qty, exc=exc,
            stop_orders=len(stop_orders))
    return p, (stop_orders[0] if stop_orders else None), exc


POS_IB, STOP_ORDER, EXC = reconcile()

## 4. Entrada con stop adjunto (bracket)

**Por qué market + stop adjunto y no otra cosa.** La entrada es a mercado porque el disparador
no es el objeto del ejercicio y una limitada podría no llenarse durante la demo; el coste de
cruzar el spread está medido y es pequeño frente a R (≈5% a 30 min). El stop va **adjunto en el
bróker**, no en este proceso, y esa es la decisión que importa: si el notebook se cae, se cuelga
la red o el kernel muere, el stop **sigue vivo en IBKR**. Un stop que solo existe en el bucle de
Python es una intención, no una protección.

`transmit=False` en el padre y `transmit=True` en el hijo hace que TWS reciba las dos patas y
las active de golpe: nunca existe una posición abierta sin su stop, ni siquiera durante unos
milisegundos.

El precio del stop se redondea a `minTick` — un stop entre ticks lo rechaza el exchange.

In [ ]:
def round_tick(px):
    """Precio al múltiplo de minTick exacto.

    El segundo round() no es cosmético: con TICK = 5e-05, round(px/TICK)*TICK
    devuelve 1.0862500000000001 y TWS rechaza la orden por precio inválido.
    """
    return round(round(px / TICK) * TICK, PX_DEC)


def enter_with_bracket(side=SIDE, qty=UNIT_QTY, account=None):
    """Market de entrada + stop a 1N adjunto, transmitidos como una sola unidad."""
    sd = 1 if side == "BUY" else -1
    account = account or tradeable_account()
    ref = float(BARS["close"].iloc[-1])
    stop_px = round_tick(ref - sd * STOP_N * N)

    parent = MarketOrder(side, qty)
    parent.orderId = ib.client.getReqId()
    parent.account = account
    parent.transmit = False            # no se activa hasta que llegue el hijo
    parent.outsideRth = not USE_RTH

    stop = StopOrder("SELL" if side == "BUY" else "BUY", qty, stop_px)
    stop.orderId = ib.client.getReqId()
    stop.account = account
    stop.parentId = parent.orderId
    stop.transmit = True               # activa el conjunto
    stop.outsideRth = True             # el stop debe poder saltar fuera de RTH

    t_parent = ib.placeOrder(contract, parent)
    t_stop = ib.placeOrder(contract, stop)
    print(f"  Enviado {side} {qty:,} {contract.localSymbol} a mercado, stop {fmt_px(stop_px)} "
          f"({STOP_N:.0f}N = {fmt_px(abs(ref-stop_px))} pts)")

    # esperar el fill del padre
    for _ in range(60):
        ib.sleep(0.5)
        if t_parent.orderStatus.status in ("Filled", "Cancelled", "ApiCancelled", "Inactive"):
            break
    st = t_parent.orderStatus
    print(f"  Estado padre: {st.status} | filled {st.filled:,} @ {st.avgFillPrice} "
          f"| remaining {st.remaining}")
    return t_parent, t_stop, stop_px


def fills_vwap(trade):
    """(vwap, cantidad, cuenta) de los fills de `trade`.

    Con fills parciales el ancla de la excursión tiene que ser el VWAP de lo
    efectivamente llenado: el riesgo en USD escala con la cantidad, pero la
    lectura en R se normaliza por una distancia de precio y no cambia.

    Devuelve además la cuenta que IBKR asignó a la ejecución — no la que se
    pidió. Es el dato que permite casar apertura con cierre: una posición solo
    está cerrada si la ejecución de cierre cae en la MISMA cuenta que la abrió.
    """
    qty = sum(f.execution.shares for f in trade.fills)
    if not qty:
        return float("nan"), 0, None
    vwap = sum(f.execution.price * f.execution.shares for f in trade.fills) / qty
    accts = {f.execution.acctNumber for f in trade.fills}
    if len(accts) > 1:
        print(f"  ⚠ fills repartidos entre {sorted(accts)}: se toma la mayoritaria.")
        by_acct = {}
        for f in trade.fills:
            by_acct[f.execution.acctNumber] = (by_acct.get(f.execution.acctNumber, 0)
                                               + f.execution.shares)
        acct = max(by_acct, key=by_acct.get)
    else:
        acct = accts.pop()
    if acct not in ACCOUNTS:
        print(f"  ⚠ el fill se asignó a {acct!r}, que no estaba en ACCOUNTS.")
        ACCOUNTS[acct] = {"id": acct, "role": "fill", "monitored": False}
    return float(vwap), int(qty), acct

In [ ]:
# ── Kill switch: una llamada aplana todo y cancela todo ────────────────────
def flatten_plan():
    """{cuenta_operable: cantidad_a_cerrar} — dónde mandar cada orden de cierre.

    Une las dos vistas: la posición reportada (que puede llegar agregada bajo el
    master) y, si esa no ve nada, el neto de ejecuciones del día por subcuenta.
    El destino SIEMPRE es una cuenta operable.
    """
    ex = executions_net(max_age=0)     # aplanar exige datos frescos, no caché
    plan = {}
    for acct, r in positions_on_contract().items():
        dest = tradeable_account(
            acct if ACCOUNTS.get(acct, {}).get("role") != "master"
            else next(iter(ex), None))
        plan[dest] = plan.get(dest, 0.0) + r["position"]
    if not plan and not _SRC_OK["positions"]:
        # Solo cuando la fuente de posición NO RESPONDIÓ. Si respondió y no ve
        # nada, no hay nada que aplanar: el neto de ejecuciones del día no baja
        # con un cierre manual y mandaría una orden contra una cuenta plana.
        print("    posiciones no disponibles: se aplana según ejecuciones del día.")
        for acct, q in ex.items():
            plan[acct] = plan.get(acct, 0.0) + q
    return {a: q for a, q in plan.items() if abs(q) > 1e-9}


def kill_switch(reason="manual"):
    """Cancela órdenes vivas y aplana el contrato en TODAS las cuentas.

    Se cancela ANTES de aplanar: si se hiciera al revés, el stop adjunto seguiría
    vivo y quedaría como una orden huérfana que puede abrir posición contraria.
    """
    print(f"  KILL SWITCH ({reason}) ...")
    for o in ib.reqOpenOrders():
        if o.contract.conId == contract.conId:
            ib.cancelOrder(o.order)
    ib.sleep(1.0)
    flat = 0
    # El plan de aplanado se construye por CUENTA OPERABLE, no por cuenta que
    # reporta. En divisas sobre este login FA la posición la publica el master
    # (DF*), que rechaza órdenes sin reparto: mandarle el cierre ahí lo dejaría
    # rechazado, el kill switch cantaría éxito y la posición seguiría abierta.
    for acct, qty_signed in flatten_plan().items():
        side = "SELL" if qty_signed > 0 else "BUY"
        o = MarketOrder(side, abs(int(qty_signed)))
        o.account = acct
        trade = ib.placeOrder(contract, o)
        flat += int(qty_signed)
        print(f"    aplanando {qty_signed:+,.0f} con {side} a mercado en {acct}")
        ib.sleep(2.0)
        if trade.orderStatus.status in ("Inactive", "Cancelled", "ApiCancelled"):
            print(f"    ⚠ el cierre quedó {trade.orderStatus.status} en {acct}: "
                  f"NO se ha aplanado. Ciérralo a mano en TWS.")
    if not flat:
        print("    nada que aplanar: ni posiciones ni ejecuciones netas en este contrato.")
    ib.sleep(2.0)
    journal("kill", regla=reason, qty=flat)
    print(f"  Kill switch completado (posición previa {flat:+d}).")
    return flat


print("  kill_switch() disponible. Ejecútala en cualquier momento para aplanar y cancelar.")

## 5. La aritmética del riesgo, y de dónde sale el umbral de MAE

**Una unidad, 1N, 1% del equity.** El tamaño se fija con `Unit_Qty = RISK_PER_UNIT × Equity / N`,
de modo que una unidad que recorre 1N produce exactamente el 1% del equity. Como `R = 1N` por
definición, **1R = 1% de la cuenta**: la pérdida al stop duro de una unidad está fijada por
construcción, no por suerte.

**Con cuatro unidades cargadas sigue siendo 1%.** Es la propiedad que hace operable la escalera
Turtle y no es obvia. Se añade una unidad cada ½N y, en cada add, **todos** los stops suben a 1N
del fill más reciente. El riesgo agregado hasta ese stop común es:

| Unidades | Fills | Stop común | Riesgo agregado |
|---|---|---|---|
| 1 | P | P − 1N | 1.0 N = **1.0%** |
| 2 | P, P+½N | P − ½N | 1.5 N = **1.5%** |
| 3 | +P+1N | P | 1.5 N = **1.5%** |
| 4 | +P+1½N | P + ½N | 1.0 N = **1.0%** |

Con la carga completa el riesgo vuelve al 1% porque las tres primeras unidades ya acumulan
recorrido a favor que compensa a la última. **El pico real es 1.5% con dos y tres unidades**, y
decir "siempre 1%" sería falso: es el número que hay que vigilar, no el de los extremos. Este
invariante está fijado con un test en `qsCrypto/portfolio/mae_test.py::TestUnitRiskArithmetic`.

**El umbral de MAE.** Se declara **antes** de abrir y no se toca: `MAE_STOP_N = 0.85 N`. No sale
de maximizar nada —eso sería ajustar a la muestra— sino de dos cotas medidas sobre la geometría
de las velas, no sobre el P&L:

- **Cota inferior 0.76 N**, el rango mediano de una vela de 30 min. Por debajo de eso el toque no
  es distinguible dentro de la propia barra: un umbral irresoluble no es una regla, es ruido con
  nombre.
- **Cota superior 1.00 N**, el stop duro. Por encima no habría "cierre temprano" que valga.

Se toma el punto medio de la banda `[0.76, 1.00)`.

**Y la evidencia en contra, que también se declara.** El backtest (§8.5 de la Parte B) calibra
la MAE sobre tres regímenes de stop inicial, porque el stop con el que se observa **trunca lo que
se puede ver**: `MAE*` se mueve entre 1.00 N y 2.10 N según con qué stop se mire — un factor
2.1×. Calibrado con el stop ancho de 2N, el que deja sobrevivir bastantes trades, sale
**`MAE* = 1.38 N`**.

Y ahí está el problema con la R de este notebook:

| Stop duro | R | MAE\* en R | ¿Cabe un cierre temprano? |
|---|---|---|---|
| **1 N** (el que se opera aquí) | 1N | **1.38 R** | **No**: el stop dispara antes que el umbral |
| 2 N | 2N | **0.69 R** | Sí, con banda `[0.38 R, 1.00 R)` |

Con `R = 1N` el stop es más estrecho que la excursión que los ganadores necesitan para
resolverse: corta ganadores y no deja hueco para nada más fino. **El umbral de 0.85 N se opera
igual, porque el enunciado pide la regla y el mecanismo tiene que existir y demostrarse en vivo,
pero la expectativa declarada de antemano es que reste.** El argumento cuantitativo apunta a
operar `R = 2N` con cierre temprano en 0.69 R, y ése es el cambio que propondría para la
siguiente iteración.

**Calibración por triple barrera.** Las distribuciones de MAE/MFE que sostienen todo esto salen
de etiquetar cada trade por la primera barrera tocada — `−1N` (stop), `+3N` (objetivo), `10
barras` (vencimiento) — que es el esquema de López de Prado aplicado como regla de ejecución en
vez de como etiqueta de un clasificador. El detalle está en §8.5 del notebook de Parte B.

## 6. Monitor en vivo

Cada ciclo actualiza la excursión con el precio actual, evalúa las cuatro reglas y escribe estado
y journal. El orden de evaluación es deliberado y **pesimista**: primero lo que cierra por
pérdida, después lo que cierra por ganancia. Si dos condiciones se cumplen en el mismo ciclo,
gana la adversa — equivocarse a favor del sistema es exactamente como se construyen bots que en
el backtest brillan y en producción no.

El stop de 1N **no lo ejecuta este bucle**: vive en el bróker. El bucle solo lo observa.

In [ ]:
def market_price(ticker):
    """Precio utilizable con datos retrasados: last, si no midpoint, si no close.

    En divisas NO hay `last`: IDEALPRO no publica trades, solo el libro. El midpoint
    es ahí el único precio real, así que se pide primero en vez de depender de que
    el NaN de `last` caiga por el guardia de isfinite.
    """
    if KIND == "forex":
        cands = (ticker.midpoint(), ticker.marketPrice(), ticker.close)
    else:
        cands = (ticker.last, ticker.marketPrice(), ticker.close)
    for v in cands:
        if v is not None and np.isfinite(v) and v > 0:
            return float(v)
    if (ticker.bid and ticker.ask and np.isfinite(ticker.bid) and np.isfinite(ticker.ask)):
        return (float(ticker.bid) + float(ticker.ask)) / 2
    return float("nan")


MAX_STALE_POLLS = 12          # ~1 min sin precio utilizable → se aborta la sesión
FLAT_GRACE_POLLS = 3          # sondeos "sin posición" tolerados antes de rendirse


def closing_execution(known_execs, side, entry_account):
    """Motivo de cierre CONFIRMADO por una ejecución, o None.

    Dos condiciones, y las dos son necesarias:

    1. Que la posición no aparezca NO demuestra que se haya cerrado: puede no
       haberse propagado todavía, o estar reportada bajo otra etiqueta. Un
       cierre deja siempre un rastro positivo — una ejecución nueva en sentido
       contrario sobre este conId. Se exige ese rastro.
    2. Esa ejecución tiene que caer en la MISMA cuenta que abrió la posición.
       Un fill contrario en otra cuenta no cierra esta posición: la deja
       abierta y además abre una nueva en la otra. Casar apertura y cierre por
       accountId es lo que distingue "se cerró mi trade" de "hay una venta por
       ahí". Si aparece un cierre descasado se avisa y NO se cuenta.
    """
    closing_side = "SLD" if side > 0 else "BOT"
    for f in ib.fills():
        if (f.execution.execId in known_execs
                or f.contract.conId != contract.conId
                or f.execution.side != closing_side):
            continue
        if f.execution.acctNumber != entry_account:
            if "mismatch_fill" not in _WARNED:
                _WARNED.add("mismatch_fill")
                print(f"  ⚠ ejecución de cierre en {f.execution.acctNumber!r} pero la "
                      f"entrada está en {entry_account!r}: NO cierra esta posición.")
            continue
        return "cierre_ejecutado"
    return None


def monitor(exc, side, entry_px, qty, entry_account=None, stop_trade=None,
            hours=SESSION_HOURS, poll=POLL_SECONDS, verbose_every=LOG_EVERY,
            max_stale=MAX_STALE_POLLS):
    """Bucle de gestión. Devuelve el motivo de salida.

    Reglas, en orden de prelación pesimista:
      1. cierre temprano por MAE   (MAE_R >= MAE_STOP_N/STOP_N)
      2. objetivo                  (MFE_R >= TAKE_N/STOP_N)
      3. barrera vertical          (MAX_HOLD_BARS × BAR_SIZE)
      4. add a ½N                  (mueve el stop de TODAS las unidades)
    El stop duro de 1N lo ejecuta el bróker; aquí solo se detecta que ocurrió.
    """
    entry_account = entry_account or ACCOUNT
    print(f"  Monitorizando {contract.localSymbol} en la cuenta {entry_account} "
          f"(de {len(ACCOUNTS)} gestionadas)")
    ticker = ib.reqMktData(contract, "", False, False)
    ib.sleep(2)
    # Foto de las ejecuciones previas: cualquier fill de cierre POSTERIOR a esto
    # es la prueba de que la posición se cerró de verdad.
    known_execs = {f.execution.execId for f in ib.fills()}
    t0 = now_srv()
    deadline_bars = MAX_HOLD_BARS * BAR_SECONDS      # BAR_SIZE, no 30 min fijos
    units, last_fill, cur_qty = 1, entry_px, qty
    i, reason, stale, flat_polls, unknown_polls = 0, None, 0, 0, 0

    try:
        while True:
            ib.sleep(poll); i += 1
            px = market_price(ticker)
            if not np.isfinite(px):
                # Sin precio no hay MAE, y una posición viva que no se mide es peor
                # que ninguna: se sale por arriba en vez de girar en silencio.
                stale += 1
                if stale >= max_stale:
                    print(f"  ⚠ {stale} sondeos sin precio utilizable "
                          f"(marketDataType={MKT_DATA_TYPE}): se aborta y se aplana.")
                    reason = "no_market_data"; break
                continue
            stale = 0
            exc.update_tick(px)
            elapsed = (now_srv() - t0).total_seconds()

            # ¿sigue viva la posición? el stop del bróker pudo saltar.
            #
            # Regla: NO se concluye "cerrada" por ausencia de posición. La sesión
            # del 2026-09-07 dio por saltado un stop a 7 segundos de la entrada,
            # con el precio a 2 pips del fill y el stop 1.5 pips más abajo: la
            # lista de posiciones salió vacía y eso se leyó como stop ejecutado.
            # No se llamó al kill switch (el motivo era "ya está cerrada") y la
            # posición se quedó viva en TWS sin nadie vigilándola. Que la lista
            # saliera vacía pudo ser el filtro de cuenta o la propagación del
            # fill; lo fatal fue el paso siguiente, tratar "no lo veo" como "no
            # está". Por eso ahora el cierre exige prueba, y la prueba es una
            # ejecución contraria en LA MISMA cuenta que abrió.
            qty_signed, live_acct, live_trade_acct, _ = live_position(prefer=entry_account)
            if qty_signed is None:
                # Las fuentes no respondieron. No se sabe si hay posición, así que
                # no se decide nada: ni cerrar ni seguir como si todo fuera bien.
                unknown_polls += 1
                print(f"  ⚠ estado de posición desconocido "
                      f"({unknown_polls}/{max_stale}): no se toma ninguna decisión.")
                if unknown_polls >= max_stale:
                    reason = "estado_desconocido"; break
                continue
            unknown_polls = 0
            live_qty = int(abs(qty_signed))
            confirmed = closing_execution(known_execs, side, entry_account)
            if confirmed:
                # El stop del bróker solo se acredita como causa si es el stop
                # de ESTA cuenta: si no, el cierre vino de otro sitio y decirlo
                # "stop_1n" falsearía el journal.
                stop_filled = (stop_trade is not None
                               and stop_trade.orderStatus.status == "Filled"
                               and getattr(stop_trade.order, "account", entry_account)
                                   == entry_account)
                reason = "stop_1n_broker" if stop_filled else confirmed
                break
            if live_qty == 0:
                flat_polls += 1
                if flat_polls < FLAT_GRACE_POLLS:
                    print(f"  ⚠ sin posición reportada ({flat_polls}/{FLAT_GRACE_POLLS}) "
                          f"y sin ejecución de cierre: puede ser propagación. Se espera.")
                    continue
                print(f"  ⚠ {flat_polls} sondeos sin posición y NINGUNA ejecución de "
                      f"cierre casada con {entry_account!r}.")
                print(f"    Cuentas consultadas: {sorted(ACCOUNTS)}")
                print( "    No se asume que el stop saltó: se sale a aplanar explícitamente.")
                reason = "posicion_no_reportada"; break
            flat_polls = 0

            mae_r, mfe_r = exc.mae_r, exc.mfe_r
            pnl_r = side * (px - exc.anchor0) / R_PTS
            # Fracción del presupuesto de MAE ya gastada: 1.00 = se cierra.
            mae_budget = mae_r / (MAE_STOP_N / STOP_N)

            state = {
                "timestamp": now_srv().isoformat(), "session": SESSION_ID,
                "symbol": SYMBOL, "contract": contract.localSymbol,
                "side": side, "qty": live_qty, "account": live_acct,
                "trade_account": live_trade_acct, "units": units,
                "price": px, "entry": exc.anchor0, "n": N, "r_pts": R_PTS,
                "pnl_r": pnl_r, "mae_r": mae_r, "mae_price": exc.mae_px,
                "mfe_r": mfe_r, "mfe_price": exc.mfe_px,
                "mae_budget": mae_budget,
                "minutes_in_trade": elapsed / 60,
                "risk_usd": live_qty * R_PTS * MULT,
                "rules": {"mae_stop_n": MAE_STOP_N, "take_n": TAKE_N,
                          "max_hold_bars": MAX_HOLD_BARS},
            }
            write_state(state)

            if i % verbose_every == 0:
                print(f"  {now_srv():%H:%M:%S} px={fmt_px(px):>12} qty={live_qty:>8,d} "
                      f"| P&L {pnl_r:+6.2f}R | MAE {mae_r:5.2f}R ({fmt_px(exc.mae_px)}) "
                      f"[{mae_budget:4.0%} del cierre] | MFE {mfe_r:5.2f}R "
                      f"({fmt_px(exc.mfe_px)}) | {elapsed/60:5.1f} min")
                journal("tick", regla="monitor", price=px, qty=live_qty, exc=exc,
                        pnl_r=pnl_r, units=units, mae_budget=mae_budget,
                        account=live_acct)

            # ── 1) cierre temprano por MAE (pesimista: se evalúa primero) ──
            if mae_r >= MAE_STOP_N / STOP_N:
                reason = "early_close_mae"; break
            # ── 2) objetivo ──
            if mfe_r >= TAKE_N / STOP_N:
                reason = "target_3n"; break
            # ── 3) barrera vertical ──
            if elapsed >= deadline_bars:
                reason = "vertical"; break
            if elapsed >= hours * 3600:
                reason = "session_end"; break

            # ── 4) add a ½N: sube el stop de TODAS las unidades ──
            trig = last_fill + side * ADD_N * N
            if units < MAX_UNITS and ((px >= trig) if side > 0 else (px <= trig)):
                o = MarketOrder("BUY" if side > 0 else "SELL", UNIT_QTY)
                o.account = entry_account      # el add va a la cuenta de la entrada
                t_add = ib.placeOrder(contract, o)
                ib.sleep(2)
                vwap, filled, add_acct = fills_vwap(t_add)
                if filled and add_acct != entry_account:
                    # Un add en otra cuenta no engorda esta posición: crea una
                    # segunda. Ni se reancla ni se mueve el stop; se deja constancia.
                    print(f"  ⚠ el ADD se ejecutó en {add_acct!r} y la posición base "
                          f"está en {entry_account!r}: son DOS posiciones, no una "
                          f"escalera. No se reancla.")
                    journal("add_descasado", regla="cuenta_distinta", price=vwap,
                            qty=filled, exc=exc, add_account=add_acct,
                            entry_account=entry_account)
                elif filled:
                    units += 1; last_fill = vwap; cur_qty += filled
                    exc.reanchor(vwap)              # solo el ancla de barreras
                    new_stop = round_tick(vwap - side * STOP_N * N)
                    for oo in ib.reqOpenOrders():
                        if (oo.contract.conId == contract.conId
                                and oo.order.orderType.startswith("STP")):
                            oo.order.auxPrice = new_stop
                            oo.order.totalQuantity = cur_qty
                            ib.placeOrder(contract, oo.order)   # modifica en sitio
                    print(f"  + ADD unidad {units} @ {fmt_px(vwap)} ({filled:,}) "
                          f"→ stop de {cur_qty:,} a {fmt_px(new_stop)}")
                    journal("add", regla=f"mfe>={ADD_N}N_desde_fill", price=vwap,
                            qty=filled, exc=exc, units=units, new_stop=new_stop)
    finally:
        ib.cancelMktData(contract)
    return reason

## 7. Ejecutar la sesión

Esta es la única celda que **abre riesgo**. Está separada a propósito: todo lo anterior se puede
ejecutar entero sin mandar una sola orden.

In [ ]:
# ⚠️ ESTA CELDA ABRE POSICIÓN REAL EN LA CUENTA PAPER
RUN_LIVE = True        

if not RUN_LIVE:
    print("  RUN_LIVE=False — no se envía nada. Ponlo a True para lanzar la sesión.")
else:
    if EXC is None:                                    # arranque limpio
        t_parent, t_stop, stop_px = enter_with_bracket()
        # ENTRY_ACCOUNT es la cuenta que IBKR asignó al fill, no la que se pidió.
        # Es la referencia contra la que se casa el cierre: sin ella, "hay una
        # venta" y "mi posición está cerrada" son indistinguibles.
        vwap, filled, ENTRY_ACCOUNT = fills_vwap(t_parent)
        if not filled:
            raise RuntimeError(f"la entrada no llenó: {t_parent.orderStatus.status}")
        if ENTRY_ACCOUNT != ACCOUNT:
            print(f"  ⚠ la entrada se ejecutó en {ENTRY_ACCOUNT!r}, no en {ACCOUNT!r}. "
                  f"El monitor sigue a {ENTRY_ACCOUNT!r}, que es donde está el riesgo.")
        sd = 1 if SIDE == "BUY" else -1
        EXC = Excursion.open(side=sd, fill=vwap, n=N, stop_n=STOP_N)
        if filled < UNIT_QTY:
            print(f"  ⚠ Fill PARCIAL {filled}/{UNIT_QTY}: el riesgo en USD baja a "
                  f"${filled*R_PTS*MULT:,.0f}, pero la lectura en R no cambia "
                  f"(se normaliza por una distancia de precio, no por cantidad).")
        journal("entry", regla="market_discrecional", price=vwap, qty=filled, exc=EXC,
                stop_px=stop_px, requested=UNIT_QTY, partial=bool(filled < UNIT_QTY),
                account=ENTRY_ACCOUNT, accounts=sorted(ACCOUNTS))
        print(f"  ENTRADA {SIDE} {filled:,} @ {fmt_px(vwap)} | stop {fmt_px(stop_px)} | "
              f"riesgo ${filled*R_PTS*MULT:,.0f}")
        side, entry_px, qty = sd, vwap, filled
        stop_trade = t_stop
    elif not ENTER_ON_RECONCILED:                      # reanudar tras caída
        side = 1 if POS_IB.position > 0 else -1
        entry_px, qty = EXC.anchor0, int(abs(POS_IB.position))
        stop_trade = STOP_ORDER
        # Al reanudar, la cuenta de referencia es donde la posición está AHORA.
        ENTRY_ACCOUNT = POS_IB.account or ACCOUNT
        print(f"  Reanudando posición existente: {qty:,} unidades en {ENTRY_ACCOUNT}, "
              f"MAE={EXC.mae_r:.2f}R")
    else:                          # entrar ENCIMA de la posición reconciliada
        prev_qty = int(abs(POS_IB.position))
        ENTRY_ACCOUNT = tradeable_account(POS_IB.account)
        sd = 1 if POS_IB.position > 0 else -1
        print(f"  Reconciliado {POS_IB.position:+,.0f} en {POS_IB.account} "
              f"(medio {fmt_px(EXC.anchor0)}). Se entra ENCIMA con {UNIT_QTY:,} más.")
        t_parent, t_stop, stop_px = enter_with_bracket(account=ENTRY_ACCOUNT)
        vwap, filled, fill_acct = fills_vwap(t_parent)
        if not filled:
            raise RuntimeError(f"la entrada no llenó: {t_parent.orderStatus.status}")
        ENTRY_ACCOUNT = fill_acct or ENTRY_ACCOUNT
        side, entry_px = sd, vwap
        qty = prev_qty + filled

        # La excursión se reancla al fill NUEVO: la MAE que se vigila es la de
        # esta entrada, no la de la posición heredada, cuyo precio medio viene de
        # ejecuciones de horas antes y mezclaría dos trades en una sola lectura.
        EXC = Excursion.open(side=sd, fill=vwap, n=N, stop_n=STOP_N)

        # El stop adjunto nació dimensionado para la parte nueva. Tiene que
        # cubrir el TOTAL: dejarlo en {filled} dejaría {prev_qty} a pelo.
        moved = 0
        for oo in ib.reqOpenOrders():
            if (oo.contract.conId == contract.conId
                    and oo.order.orderType.startswith("STP")):
                oo.order.totalQuantity = qty
                ib.placeOrder(contract, oo.order)
                moved += 1
        ib.sleep(1.0)
        print(f"  ENTRADA {SIDE} {filled:,} @ {fmt_px(vwap)} | total {qty:,} | "
              f"stop {fmt_px(stop_px)} ({STOP_N:.0f}N) ampliado en {moved} orden(es)")
        journal("entry", regla="encima_de_reconciliada", price=vwap, qty=filled,
                exc=EXC, stop_px=stop_px, requested=UNIT_QTY, previous_qty=prev_qty,
                total_qty=qty, account=ENTRY_ACCOUNT, stops_resized=moved)
        stop_trade = t_stop

    # stop_trade importa: es la prueba positiva de que un stop se ejecutó, frente
    # a la mera ausencia de posición en el feed.
    reason = monitor(EXC, side, entry_px, qty, entry_account=ENTRY_ACCOUNT,
                     stop_trade=stop_trade)
    print(f"\n  Motivo de salida: {reason}")

    if reason not in ("stop_1n_broker",):
        flat = kill_switch(reason)
    # El snapshot puede volver vacío (mercado cerrado, o el motivo de salida fue
    # justamente quedarse sin datos): en ese caso se registra el último precio que
    # SÍ se midió, no un NaN que rompería el JSON del journal.
    exit_px = market_price(ib.reqMktData(contract, "", True, False))
    if not np.isfinite(exit_px):
        exit_px = float(EXC.anchor)
        print("  ⚠ Sin snapshot de precio a la salida: se journaliza el último ancla.")
    journal("exit", regla=reason, price=exit_px, qty=qty, exc=EXC,
            account=ENTRY_ACCOUNT)
    print(f"  MAE final {EXC.mae_r:.2f}R ({fmt_px(EXC.mae_px)}) | MFE {EXC.mfe_r:.2f}R")

## 8. Journal de la sesión

In [ ]:
# Lectura del journal como tabla: es el entregable que se adjunta.
rows = [json.loads(l) for l in open(JOURNAL)] if JOURNAL.exists() else []
J = pd.DataFrame(rows)
if len(J):
    cols = [c for c in ["ts", "evento", "regla", "price", "qty", "units",
                        "pnl_r", "mae_r", "mae_price", "mfe_r"] if c in J.columns]
    display(J[cols])
    dec = J[J.evento != "tick"]
    print(f"\n  {len(J)} líneas ({len(dec)} decisiones, {len(J)-len(dec)} ticks)")
    print(f"  Eventos: {J.evento.value_counts().to_dict()}")
    csv = JOURNAL.with_suffix(".csv")
    J.to_csv(csv, index=False)
    print(f"  Exportado: {csv}")
else:
    print("  Journal vacío: la sesión no llegó a operar.")

## 9. CPPI como propuesta de gestión de riesgo

Lo de arriba dimensiona por **volatilidad**: `Unit_Qty = 1%·Equity / N`. Eso responde "¿cuánto
me puede doler una unidad?", pero no responde "¿cuánto puedo permitirme perder **en total**
antes de que la cuenta deje de ser recuperable?". CPPI (*Constant Proportion Portfolio
Insurance*) responde justo a esa segunda pregunta, y por eso son complementarias en vez de
alternativas.

El pipeline, tal como está implementado en el market maker de este mismo repo:

```
W        = min(equity_actual, capital_inicial)
floor    = floor_fraction × capital_inicial        # el suelo que se protege
cushion  = max(W − floor, 0)                       # lo que se puede arriesgar
B        = m_cppi × cushion                        # presupuesto de riesgo
R_unit   = mid × σ × √dt                           # riesgo por unidad
Qmax     = min(B / R_unit, B / mid)                # el más estricto de riesgo y nocional
c        = cushion / W
Qmax_eff = Qmax × c^cushion_exponent               # escalado potencial del colchón
```

**La propiedad que importa:** cuando el equity se acerca al suelo, `cushion → 0` y por tanto
`Qmax_eff → 0`. El tamaño se apaga **solo**, de forma continua, sin necesidad de una regla
discreta de "si pierdo X%, paro". El `cushion_exponent` gradúa la agresividad: 0.3 desapalanca
tarde y brusco, 2.0 desapalanca pronto y suave.

**Cómo encaja con lo de este notebook.** Turtle/MAE acota el riesgo **por trade**; CPPI acota el
riesgo **de la cuenta**. El tamaño operable es el **mínimo de los dos**:

```
qty = min( Unit_Qty_Turtle , Qmax_eff_CPPI )
```

que es exactamente cómo `TurtlePositionLimits` recorta `CPPIState.Qmax_eff` en el sistema de
market making (skill `turtlerisk`, §3.3). Ninguno sustituye al otro: sin Turtle, CPPI no sabe
cuánto se mueve *este* instrumento; sin CPPI, Turtle puede perder el 1% cuarenta veces seguidas
y llevarse el 40% de la cuenta sin que ninguna regla se active.

La regla de drawdown Turtle (−10% de equity ⇒ −20% de nocional) es una versión discreta y tosca
de lo mismo. CPPI la generaliza a una función continua con un suelo explícito, que es lo que
propondría para producción.

In [ ]:
# Cierre de sesión: TWS mantiene ocupado el clientId mientras el socket viva.
journal("session_end", regla="cierre")
if ib.isConnected():
    ib.disconnect()
    print("  Sesión IBKR cerrada.")